In [ ]:
import altair as alt
import pandas as pd
from omegaconf import OmegaConf

from update_vars import GTFS_DATA_DICT, RT_SCHED_GCS, SCHED_GCS, SEGMENT_GCS

In [ ]:
readable_dict = OmegaConf.load("readable2.yml")

In [ ]:
subset = [
    "direction_id",
    "time_period",
    "avg_scheduled_service_minutes",
    "n_scheduled_trips",
    "service_date",
    "recent_combined_name",
    "route_primary_direction",
    "minutes_atleast1_vp", 
    "minutes_atleast2_vp",
    "is_early",
    "is_ontime",
    "is_late",
    "vp_per_minute",
    "pct_in_shape",
    "pct_sched_journey_atleast1_vp",
    "pct_sched_journey_atleast2_vp",
    "rt_sched_journey_ratio",
    "speed_mph",
    "portfolio_organization_name",
    "headway_in_minutes",
 "sched_rt_category" # added this
]

In [ ]:
readable_col_names = {
    "direction_id": "Direction (0/1)",
    "time_period": "Period",
    "avg_scheduled_service_minutes": "Average Scheduled Service (trip minutes)",
    "n_scheduled_trips": "# Scheduled Trips",
    "service_date": "Date",
    "recent_combined_name": "Route",
    "route_primary_direction": "Direction",
    "minutes_atleast1_vp": "# Minutes with 1+ VP per Minute",
    "minutes_atleast2_vp": "# Minutes with 2+ VP per Minute",
    "is_early": "# Early Arrival Trips",
    "is_ontime": "# On-Time Trips",
    "is_late": "# Late Trips",
    "vp_per_minute": "Average VP per Minute",
    "pct_in_shape": "% VP within Scheduled Shape",
    "pct_sched_journey_atleast1_vp": "% Scheduled Trip w/ 1+ VP/Minute",
    "pct_sched_journey_atleast2_vp": "% Scheduled Trip w/ 2+ VP/Minute",
    "rt_sched_journey_ratio": "Realtime versus Scheduled Service Ratio",
    "speed_mph": "Speed (MPH)",
    "portfolio_organization_name": "Portfolio Organization Name",
    "headway_in_minutes": "Headway (Minutes)",
}

In [ ]:
import stuff_to_add

FILE = GTFS_DATA_DICT.digest_tables.route_schedule_vp

# some of the portfolio grain can be dealt with
# but separate out the renaming/replacing/subsetting to separate script

df = pd.read_parquet(
    f"{RT_SCHED_GCS}{FILE}.parquet",
    filters = [[
        ("portfolio_organization_name", "==", "City of West Hollywood"),
        ("recent_combined_name", "in", ["Cityline Local-East", "Cityline Local-West"])
    ]]
).pipe(
    stuff_to_add.data_wrangling_for_visualizing2,
    subset, readable_col_names
)

In [ ]:
import _report_visuals_utils
from IPython.display import HTML, display

# Set drop down menu to be on the upper right for the charts
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [ ]:
def route_filter(df):
    routes_list = df["Route"].unique().tolist()

    route_dropdown = alt.binding_select(
        options=routes_list,
        name="Routes: ",
    )
    # Column that controls the bar charts
    xcol_param = alt.selection_point(
        fields=["Route"], value=routes_list[0], bind=route_dropdown
    )
    
    
    chart1 = (
        stuff_to_add.sample_spatial_accuracy_chart(df[df.Period=="All Day"])
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    ) 

    chart2 = (
        stuff_to_add.sample_avg_scheduled_min_chart(df[df.Period=="All Day"])
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    
    )
    
    chart_list = [
        chart1,
        chart2
    ]
    chart = alt.vconcat(*chart_list)

    
    return chart
 

In [ ]:
route_filter(df)